<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Solution: Bike-share demand and fleet planning

**[Bike-share demand and fleet planning](https://learning.nextia-ai.com/courses/forecasting/m09/bike-share-demand-and-fleet-planning/)** · Forecasting and Time Series: Predict What Comes Next · Case study 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** forecast a real bike-share system's rentals 14 days ahead per day and one week ahead per hour, compare the seasonal naive forecast, ETS, MSTL and LightGBM in the course's backtest, measure what the weather would be worth if you knew it, and set the bikes and the rebalancing crews at a quantile chosen from their costs.

| | |
|---|---|
| **Time** | About 2.5 hours with the lesson page; the notebook runs in about 2 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | The backtest and WAPE, from [Test like the future](https://learning.nextia-ai.com/courses/forecasting/m03/test-like-the-future-backtesting/) and [Measure the error that matters](https://learning.nextia-ai.com/courses/forecasting/m03/measure-the-error-that-matters/). Known future inputs, from [Calendars, holidays and events](https://learning.nextia-ai.com/courses/forecasting/m02/calendars-holidays-and-events/). LightGBM, from [Gradient boosting with LightGBM](https://learning.nextia-ai.com/courses/forecasting/m05/gradient-boosting-with-lightgbm/). Plans at a quantile, from [From forecast to order and roster](https://learning.nextia-ai.com/courses/forecasting/m07/from-forecast-to-order-and-roster/). |
| **Data** | UCI Bike Sharing, CC BY 4.0: real rentals per day and per hour of the Capital Bikeshare system in Washington, D.C., 2011 and 2012, with the weather |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0 and LightGBM 4.7.0, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m09/bike-share-demand-and-fleet-planning/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M09-L01-bike-share-demand-and-fleet-planning/bike-share-demand-and-fleet-planning-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, lightgbm 4.7.0, scikit-learn 1.9.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 mlforecast==1.1.0 lightgbm==4.7.0 scikit-learn==1.9.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)
import mlforecast; print("mlforecast", mlforecast.__version__)
import lightgbm; print("lightgbm", lightgbm.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the data

Run the next cell. It downloads two files into a new folder `bike-share-forecasting` and checks each file's checksum:

- `bike+sharing+dataset.zip` (280 KB): two years of rentals of a real bike-share system, from the UCI Machine Learning Repository. It holds `day.csv` and `hour.csv`;
- `backtest.py`: the course's backtest helper, the same file as in every lesson of the course.

It needs an internet connection and no account, and takes about 10 seconds. You should see `Ready: day.csv (731 rows), hour.csv (17379 rows) and backtest.py.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `bike-share-forecasting` (in Colab: the **Files** panel on the left), then run the cell again. If it happens again, the source file has changed: tell us (the link is at the end of the notebook).

**Source and licence.** Fanaee-T, H. (2013). *Bike Sharing* [Dataset]. UCI Machine Learning Repository. <https://doi.org/10.24432/C5W894>. Licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The notebook reads the original files and does not change them.

In [ ]:
# Setup: download the data and the backtest helper into bike-share-forecasting/ and check them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

FILES = {
    "bike+sharing+dataset.zip": [
        "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip",
        "b70182d0d0508e9abbb79306ce5c0cec34869000f8220175ac83d11dbe845401"
    ],
    "backtest.py": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/backtest.py",
        "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a"
    ]
}
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("bike-share-forecasting").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, (url, expected) in FILES.items():
    path = PROJECT / name
    if path.exists() and sha256(path) == expected:
        continue
    if (HERE / name).exists():  # a copy next to the notebook
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder bike-share-forecasting and run again.")

os.chdir(PROJECT)
with zipfile.ZipFile("bike+sharing+dataset.zip") as z:
    z.extract("day.csv")
    z.extract("hour.csv")
rows = {f: sum(1 for _ in open(f)) - 1 for f in ["day.csv", "hour.csv"]}
print(f"Ready: day.csv ({rows['day.csv']} rows), hour.csv ({rows['hour.csv']} rows) and backtest.py.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The rentals

`day.csv` has one row per day, and `hour.csv` one row per hour. Both count the bikes rented in the whole system (`cnt`), with the calendar and the observed weather. The next cell loads both files and runs the first checks.

In [ ]:
day = pd.read_csv("day.csv", parse_dates=["dteday"])
hour = pd.read_csv("hour.csv", parse_dates=["dteday"])
hour["ds"] = hour["dteday"] + pd.to_timedelta(hour["hr"], unit="h")
every_hour = pd.date_range(hour["ds"].min(), hour["ds"].max(), freq="h")
missing = every_hour.difference(hour["ds"])
same = bool((hour.groupby("dteday")["cnt"].sum().to_numpy() == day["cnt"].to_numpy()).all())
print(day.shape, hour.shape, "| days:", day["dteday"].min().date(), "to", day["dteday"].max().date())
print(f"missing hours: {len(missing)} | the days add up the hours: {same}")
print("days with the most missing hours:", {str(k): v for k, v in pd.Series(missing.date).value_counts().head(3).items()})
print("lowest days:", day.nsmallest(3, "cnt")[["dteday", "cnt"]].to_string(index=False))

> **Check.** You should see 731 days and 17,379 hours, from 2011-01-01 to 2012-12-31, and 165 missing hours. The hours add up to the days exactly. The lowest day is 29 October 2012, with 22 rentals: Hurricane Sandy closed the system, and most of its hours have no row.

Two columns, `casual` and `registered`, add up to `cnt`. They are counted in the same hour, so they are a leak, and we do not use them. The machine learning course's case study *Forecast bike-share demand* shows this leak.

## 2. Two series: per day and per hour

The backtest needs the long format: `unique_id`, `ds`, `y`. An hour with no row had no rental, so it gets 0. Its weather is the weather of the hour before. The working-day and holiday flags of an hour come from its day.

In [ ]:
WEATHER = ["temp", "hum", "windspeed", "weathersit"]
CALENDAR = ["workingday", "holiday"]
daily = pd.DataFrame({"unique_id": "rentals", "ds": day["dteday"], "y": day["cnt"].astype(float)})
daily[CALENDAR + WEATHER] = day[CALENDAR + WEATHER].to_numpy()
hourly = pd.DataFrame({"ds": every_hour}).merge(hour[["ds", "cnt"] + WEATHER], on="ds", how="left")
hourly["y"] = hourly["cnt"].fillna(0).astype(float)  # a missing hour had no rental
hourly[WEATHER] = hourly[WEATHER].ffill()
cal = day.set_index("dteday")[CALENDAR]
for c in CALENDAR:
    hourly[c] = hourly["ds"].dt.normalize().map(cal[c]).astype(int)
hourly = hourly.assign(unique_id="rentals")[["unique_id", "ds", "y"] + CALENDAR + WEATHER]
print(len(daily), "days |", len(hourly), "hours | rentals:", int(daily["y"].sum()), int(hourly["y"].sum()))

> **Check.** You should see `731 days | 17544 hours` (731 × 24) and the same number of rentals twice, 3,292,679.

## 3. What matters for the plan

Three things: the growth, the shape of a day, and the weather.

> **Predict.** The system grew a lot from 2011 to 2012. By how much, roughly: +10%, +30% or +60%? Then run the cell.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(day["dteday"], day["cnt"], lw=0.8)
ax.set_ylabel("rentals per day")
ax.set_title("Rentals per day, 2011 and 2012")
plt.show()

w = hourly.assign(hr=hourly["ds"].dt.hour)
shape = w.groupby(["workingday", "hr"])["y"].mean().unstack(0).round(0)
print("mean rentals per hour (working day = 1):")
print(shape.loc[[6, 8, 13, 17, 22]].to_string())
d2 = day.assign(m=day["dteday"].dt.to_period("M"))
d2["vs_month"] = 100 * (d2["cnt"] / d2.groupby("m")["cnt"].transform("mean") - 1)
print("rentals against the month's mean, by weather type (1 clear, 2 mist, 3 rain or snow), %:")
print(d2.groupby("weathersit")["vs_month"].agg(["mean", "size"]).round(1).to_string())

> **Check.** You should see a yearly wave, higher in 2012. Working days have two peaks (473 rentals at 08:00, 524 at 17:00); other days have one wide peak around 13:00 (373). A day with rain or snow has 52.5% fewer rentals than its month's mean (21 days).

> **Your turn.** By how many percent did the mean rentals **per day** grow from 2011 to 2012? Put the number in `growth_pct`, rounded to 1 decimal. Then run the check cell.

In [ ]:
year_mean = day.groupby(day["dteday"].dt.year)["cnt"].mean()
growth_pct = round(float(100 * (year_mean[2012] / year_mean[2011] - 1)), 1)
print(year_mean.round(0).to_dict(), growth_pct)

In [ ]:
expect('growth_pct', growth_pct, 64.4, tolerance=0.0005)

## 4. The two decisions and their costs

Amara plans two things on Sunday evening. Every number below is an **assumption** of this case study, not the operator's.

- **The fleet, for the next 14 days:** the bikes in service each day. A bike in service makes at most 5 rentals a day. A bike short for a day loses 5 rentals at about 8 dollars each: **40 dollars**. An extra bike in service costs **10 dollars** a day (checks, cleaning, a place at a station). The critical ratio is 40 ÷ (40 + 10) = **0.8**.
- **The rebalancing crews, for the next week:** a crew (a van and a driver) keeps up with 100 rentals an hour, from 06:00 to 22:59. An extra crew hour costs **45 dollars**. An hour one crew short costs **135 dollars** (empty and full stations, lost rentals). The critical ratio is 135 ÷ (135 + 45) = **0.75**.

The crew cost is the course's `bt.roster_cost()` with these numbers. The fleet cost has the same shape, per day.

In [ ]:
RENTALS_PER_BIKE, BIKE_SHORT, BIKE_EXTRA = 5, 40.0, 10.0
RENTALS_PER_CREW, CREW_EXTRA, CREW_SHORT, CREW_HOURS = 100, 45.0, 135.0, range(6, 23)
FLEET_RATIO = BIKE_SHORT / (BIKE_SHORT + BIKE_EXTRA)
CREW_RATIO = CREW_SHORT / (CREW_SHORT + CREW_EXTRA)
from scipy.stats import norm

def fleet_cost(fc, column="yhat"):
    """The cost of a 14-day fleet plan, per origin: bikes in service each day = plan / 5, rounded up."""
    bikes = np.ceil(fc[column].clip(lower=0) / RENTALS_PER_BIKE)
    need = np.ceil(fc["y"] / RENTALS_PER_BIKE)
    cost = np.maximum(need - bikes, 0) * BIKE_SHORT + np.maximum(bikes - need, 0) * BIKE_EXTRA
    return float(cost.sum() / fc["origin"].nunique())

crew_cost = bt.roster_cost(per_agent=RENTALS_PER_CREW, agent_hour=CREW_EXTRA, short_hour=CREW_SHORT,
                           open_hours=CREW_HOURS)

def at_quantile(fc, q):
    """The plan at the q quantile: the forecast plus z(q) spreads, the spread from the 80% range (normal)."""
    sd = (fc["q90"] - fc["q10"]) / (2 * norm.ppf(0.9))
    return (fc["yhat"] + norm.ppf(q) * sd).clip(lower=0)

print("critical ratios: fleet", FLEET_RATIO, "| crews", CREW_RATIO)

> **Check.** You should see `critical ratios: fleet 0.8 | crews 0.75`.

## 5. The forecasters

The course backtest, on this data: **13 cut-offs** (the code calls them origins), 28 days apart, every fourth Sunday from 18 December 2011 to 18 November 2012, refit at every cut-off. Per day, 14 days ahead; per hour, 168 hours ahead from 23:00 on the same Sundays. Each forecaster returns its forecast and its 80% range (`q10`, `q90`).

- **Seasonal naive:** the same weekday last week (per day), the same hour last week (per hour). The course's baseline.
- **ETS** with a weekly season (per day), and **MSTL** with a day and a week season on the last 12 weeks (per hour), as in [Several seasons and long series](https://learning.nextia-ai.com/courses/forecasting/m04/several-seasons-and-long-series/).
- **LightGBM, no weather:** inputs that Amara has on Sunday evening: past rentals, the weekday, the month, the working-day and holiday flags. Per day, every lag is at least 14 days old, so it is known for all 14 days. Its range comes from 4 earlier windows (conformal).
- **LightGBM, real weather (impossible):** the same, plus the **observed** weather of each future day or hour. Nobody has that on Sunday evening. It shows what a perfect weather forecast would be worth.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import MSTL, AutoETS, SeasonalNaive
import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from mlforecast.utils import PredictionIntervals

ORIGINS = bt.origins("2012-11-18", n=13, every="28D")       # Sundays, 18 Dec 2011 to 18 Nov 2012
HOURLY_ORIGINS = [o + pd.Timedelta(hours=23) for o in ORIGINS]
FINAL_ORIGIN = pd.Timestamp("2012-12-02")

def stats(make_model, freq, tail=None):
    """A statsforecast model as a forecaster, with its 80% range as q10 and q90."""
    def forecaster(train, h, future):
        t = train if tail is None else train.tail(tail)
        model = make_model()
        f = StatsForecast(models=[model], freq=freq).forecast(df=t[["unique_id", "ds", "y"]], h=h, level=[80])
        n = str(model)
        out = pd.DataFrame({"unique_id": f["unique_id"], "ds": f["ds"], "yhat": f[n],
                            "q10": f[f"{n}-lo-80"], "q90": f[f"{n}-hi-80"]})
        return out.assign(**{c: out[c].clip(lower=0) for c in ["yhat", "q10", "q90"]})
    return forecaster

def lightgbm(freq, inputs, lags, lag_transforms, date_features, params):
    """LightGBM with mlforecast, with an 80% range from 4 earlier windows (conformal)."""
    def forecaster(train, h, future):
        m = MLForecast(models={"lgbm": lgb.LGBMRegressor(**params)}, freq=freq, lags=lags,
                       lag_transforms=lag_transforms, date_features=date_features, num_threads=1)
        m.fit(train[["unique_id", "ds", "y"] + inputs], static_features=[],
              prediction_intervals=PredictionIntervals(n_windows=4, h=h))
        p = m.predict(h, X_df=future[["unique_id", "ds"] + inputs], level=[80])
        out = p.rename(columns={"lgbm": "yhat", "lgbm-lo-80": "q10", "lgbm-hi-80": "q90"})
        return out.assign(**{c: out[c].clip(lower=0) for c in ["yhat", "q10", "q90"]})
    return forecaster

LGB_DAY = dict(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=10, subsample=0.8,
               subsample_freq=1, colsample_bytree=0.8, random_state=35, n_jobs=2, verbose=-1)
LGB_HOUR = dict(LGB_DAY, num_leaves=31, min_child_samples=20)
DAILY = {
    "seasonal naive (7 days)": stats(lambda: SeasonalNaive(season_length=7), "D"),
    "ETS (weekly season)": stats(lambda: AutoETS(season_length=7), "D"),
    "LightGBM, no weather": lightgbm("D", CALENDAR, [14, 21, 28], {14: [RollingMean(7), RollingMean(28)]},
                                     ["dayofweek", "month", "dayofyear"], LGB_DAY),
    "LightGBM, real weather (impossible)": lightgbm("D", CALENDAR + WEATHER, [14, 21, 28],
                                                    {14: [RollingMean(7), RollingMean(28)]},
                                                    ["dayofweek", "month", "dayofyear"], LGB_DAY),
}
HOURLY = {
    "seasonal naive (168 hours)": stats(lambda: SeasonalNaive(season_length=168), "h"),
    "MSTL (day and week)": stats(lambda: MSTL(season_length=[24, 168], trend_forecaster=AutoETS(model="ZZN")),
                                 "h", tail=2016),
    "LightGBM, no weather": lightgbm("h", CALENDAR, [1, 24, 168], {24: [RollingMean(24)], 168: [RollingMean(168)]},
                                     ["hour", "dayofweek", "month"], LGB_HOUR),
    "LightGBM, real weather (impossible)": lightgbm("h", CALENDAR + WEATHER, [1, 24, 168],
                                                    {24: [RollingMean(24)], 168: [RollingMean(168)]},
                                                    ["hour", "dayofweek", "month"], LGB_HOUR),
}
print(len(ORIGINS), "origins:", ORIGINS[0].date(), "to", ORIGINS[-1].date())

> **Check.** You should see `13 origins: 2011-12-18 to 2012-11-18`: the 13 cut-offs.

## 6. The backtest

The next cell runs every forecaster at the 13 cut-offs. For each one, it scores the forecast with the course's measures, and the plan twice: at the forecast itself, and at the cost quantile (0.8 for the fleet, 0.75 for the crews). `80% holds` is the coverage of the 80% range.

> **Predict.** The weather moves rentals by half on a rainy day. How much better will LightGBM be **without** the weather than the seasonal naive forecast? Then run the cell (about 45 seconds).

In [ ]:
def run(methods, df, h, origins, season, cost, ratio):
    rows, fcs = [], {}
    for name, f in methods.items():
        fc = bt.backtest(df, f, h=h, origins=origins)
        fc["plan"] = at_quantile(fc, ratio)
        row = bt.score(fc, df, name, season=season, cost=cost)
        row["cost at quantile"] = round(cost(fc, "plan"))
        row["80% holds"] = bt.coverage(fc, "q10", "q90")
        rows.append(row)
        fcs[name] = fc
    t = pd.DataFrame(rows).rename(columns={"cost $": "cost at forecast"})
    t = t[["method", "MASE", "WAPE %", "bias %", "pinball", "80% holds", "cost at forecast", "cost at quantile"]]
    print(t.to_string(index=False))
    return t, fcs

print("PER DAY, 14 days ahead: fleet cost per 14-day plan, dollars")
daily_table, daily_fc = run(DAILY, daily, 14, ORIGINS, 7, fleet_cost, FLEET_RATIO)
print("\nPER HOUR, one week ahead: crew cost per week, dollars")
hourly_table, hourly_fc = run(HOURLY, hourly, 168, HOURLY_ORIGINS, 168, crew_cost, CREW_RATIO)

> **Check.** Per day: the three honest methods have almost the same WAPE (17.8% to 18.1%), and LightGBM without weather has the lowest cost at the quantile (48,474 dollars per 14-day plan). With the real weather, WAPE falls to 15.2%. Per hour: LightGBM without weather has the lowest cost at the quantile (5,853 dollars a week). MSTL's 80% range holds 0.992 of the real hours: far too wide. So its plan at the quantile is the most expensive (9,768).

## 7. One measure for the choice

Amara pays for both plans every week. So the measure is **the cost of one week**: half a 14-day fleet plan plus one week of crews, both at the cost quantile. The next cell ranks every pair of honest methods, over the 13 cut-offs.

In [ ]:
honest_d = [m for m in DAILY if "impossible" not in m]
honest_h = [m for m in HOURLY if "impossible" not in m]
fq = daily_table.set_index("method")["cost at quantile"]
cq = hourly_table.set_index("method")["cost at quantile"]
week = pd.DataFrame([{"per day": d, "per hour": h, "fleet": round(fq[d] / 2), "crews": int(cq[h]),
                      "per week": round(fq[d] / 2 + cq[h])} for d in honest_d for h in honest_h])
print(week.sort_values("per week").to_string(index=False))
impossible = "LightGBM, real weather (impossible)"
print("with the real weather (impossible):", round(fq[impossible] / 2 + cq[impossible]), "dollars a week")

> **Check.** You should see LightGBM for both plans first, at 30,090 dollars a week, and the seasonal naive pair at 34,983. With the real weather, it would be 23,826: a perfect weather forecast would be worth about 6,300 dollars a week. Nobody has one 14 days ahead.

> **Your turn.** What does a week cost with the **seasonal naive** forecast for both plans? Read it from `week` (or compute it from `fq` and `cq`), and put it in `week_naive`. Then run the check cell.

In [ ]:
week_naive = int(week[(week["per day"] == "seasonal naive (7 days)") & (week["per hour"] == "seasonal naive (168 hours)")]["per week"].iloc[0])
print(week_naive)

In [ ]:
expect('week_naive', week_naive, 34983)

Now check the quantile itself. The next cell moves the quantile of the chosen LightGBM plans from 0.5 to 0.9.

In [ ]:
QS = [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9]
lgb_d, lgb_h = daily_fc["LightGBM, no weather"], hourly_fc["LightGBM, no weather"]
curve = pd.DataFrame({"quantile": QS,
                      "fleet per 14 days": [round(fleet_cost(lgb_d.assign(p=at_quantile(lgb_d, q)), "p")) for q in QS],
                      "crews per week": [round(crew_cost(lgb_h.assign(p=at_quantile(lgb_h, q)), "p")) for q in QS]})
print(curve.to_string(index=False))

> **Check.** The fleet cost is lowest at 0.8 (48,474), the critical ratio. The crew cost is lowest at 0.7 (5,819), close to 0.75 (5,853). Both plans at the forecast itself (0.5) cost much more: 72,882 and 7,321.

> **Your turn.** A manager wants the crews at the 0.9 quantile "to be safe". What does a week of crews cost then? Put it in `crew_90`. Then run the check cell.

In [ ]:
crew_90 = round(crew_cost(lgb_h.assign(p=at_quantile(lgb_h, 0.9)), "p"))
print(crew_90)

In [ ]:
expect('crew_90', crew_90, 6767)

## 8. The final test, once

The choice is made: LightGBM without weather for both plans, at the cost quantiles. Now the 14 days after 2 December 2012, which no step has seen. We run every method once and change nothing after.

In [ ]:
fin = []
for name, f in DAILY.items():
    fc = bt.backtest(daily, f, h=14, origins=[FINAL_ORIGIN])
    fc["plan"] = at_quantile(fc, FLEET_RATIO)
    r = bt.score(fc, daily, name, season=7)
    fin.append({"plan": "fleet, 14 days", "method": name, "WAPE %": r["WAPE %"], "bias %": r["bias %"],
                "cost at quantile": round(fleet_cost(fc, "plan"))})
for name, f in HOURLY.items():
    fc = bt.backtest(hourly, f, h=168, origins=[FINAL_ORIGIN + pd.Timedelta(hours=23)])
    fc["plan"] = at_quantile(fc, CREW_RATIO)
    r = bt.score(fc, hourly, name, season=168)
    fin.append({"plan": "crews, 1 week", "method": name, "WAPE %": r["WAPE %"], "bias %": r["bias %"],
                "cost at quantile": round(crew_cost(fc, "plan"))})
print(pd.DataFrame(fin).to_string(index=False))

> **Check.** For the fleet, LightGBM is 21.5% too low and costs 55,250 dollars; ETS costs 32,460. For the crews, LightGBM is the cheapest (5,490). The chosen daily method **lost** the final test.

LightGBM learned December from December 2011, when the system was much smaller (3,210 rentals a day in the first half of December 2011, 5,266 in 2012). A tree cannot forecast above the values it has seen, as [Gradient boosting with LightGBM](https://learning.nextia-ai.com/courses/forecasting/m05/gradient-boosting-with-lightgbm/) says. We report this result and change nothing now.

## 9. The plan for Christmas week, on new input

The last step runs the chosen pipeline on all the data to Sunday 23 December 2012, and plans the next week. Then the real week arrives.

In [ ]:
NEW = pd.Timestamp("2012-12-23")
dfc = DAILY["LightGBM, no weather"](daily[daily["ds"] <= NEW], 7,
                                    daily[(daily["ds"] > NEW) & (daily["ds"] <= NEW + pd.Timedelta(days=7))].drop(columns="y"))
dfc["plan"] = at_quantile(dfc, FLEET_RATIO)
dfc = dfc.merge(daily[["ds", "y", "holiday"]], on="ds")
dfc["bikes"] = np.ceil(dfc["plan"] / RENTALS_PER_BIKE).astype(int)
dfc["bikes needed"] = np.ceil(dfc["y"] / RENTALS_PER_BIKE).astype(int)
print(dfc[["ds", "yhat", "bikes", "y", "bikes needed", "holiday"]].round(0).to_string(index=False))
print("rentals: forecast", round(dfc["yhat"].sum()), "| real", int(dfc["y"].sum()),
      "| fleet cost of the week:", round(fleet_cost(dfc.assign(origin=NEW), "plan")))

> **Check.** You should see a forecast of 25,617 rentals and 10,720 real ones. The plan has about 1,060 bikes on most days; about 200 were needed on 24 and 25 December. The week's fleet costs 46,140 dollars. Only 25 December is a holiday in the data, but the whole week is quiet.

A planner knows last Christmas. The week of 24 to 30 December 2011 had 54.3% of the mean week of the two weeks before it. The next cell applies the same share to the plan.

In [ ]:
xmas11 = day[(day["dteday"] >= "2011-12-24") & (day["dteday"] <= "2011-12-30")]["cnt"].sum()
before11 = day[(day["dteday"] >= "2011-12-10") & (day["dteday"] <= "2011-12-23")]["cnt"].sum() / 2
share = xmas11 / before11
before12 = daily[(daily["ds"] >= "2012-12-10") & (daily["ds"] <= "2012-12-23")]["y"].sum() / 2
rule = dfc.assign(plan=dfc["plan"] * share / (dfc["yhat"].sum() / before12))
print(f"last year's share: {share:.3f} | bikes: {int(np.ceil(rule['plan'] / RENTALS_PER_BIKE).sum())} "
      f"| fleet cost of the week: {round(fleet_cost(rule.assign(origin=NEW), 'plan'))}")

> **Check.** You should see `last year's share: 0.543`, 4,493 bike-days (instead of 6,761) and a cost of 23,460 dollars: half of the plan without the rule. The real week had only 34.2%: it also rained on 26 December.

## 10. Failure case: a range that is too wide

MSTL's point forecast per hour is better than the seasonal naive one, but its plan at the quantile is the most expensive. Look at the width of its range.

In [ ]:
for name in ["MSTL (day and week)", "LightGBM, no weather"]:
    f = hourly_fc[name]
    print(f"{name:22} mean width of the 80% range: {(f['q90'] - f['q10']).mean():6.1f} rentals an hour | "
          f"holds {bt.coverage(f, 'q10', 'q90')} | crews per week at the forecast {round(crew_cost(f))}, at 0.75 {round(crew_cost(f, 'plan'))}")

> **Check.** MSTL's 80% range is 574.4 rentals an hour wide on average, and holds 0.992 of the real hours. LightGBM's is 151.9 wide and holds 0.700: too narrow, but much closer. A plan at a quantile trusts the range: **score the range before you plan from it**, as in [Score the range](https://learning.nextia-ai.com/courses/forecasting/m07/score-the-range/).

## Next

You have taken two years of real rentals to a fleet plan and a crew roster with their costs. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m09/bike-share-demand-and-fleet-planning/) has the charts, the recommendation, the limits, how to make this a portfolio piece, and the knowledge checks.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Electricity load for many customers](https://learning.nextia-ai.com/courses/forecasting/m09/electricity-load-for-many-customers/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m09/bike-share-demand-and-fleet-planning/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real and anonymous: UCI Bike Sharing, CC BY 4.0.